# 02 — Build the driver-race dataset

## Purpose

Build the canonical one-row-per-driver-per-race analytical dataset for 2004–2025. Preserve every race-result row, attach qualifying and teammate information, classify participation and retirement outcomes, and create transparent model-eligibility and field-relative features.

## Inputs

- `data/raw/race_schedule.csv`
- `data/raw/race_results.csv`
- `data/raw/qualifying_results.csv`

`race_results.csv` defines the output population. Qualifying-only entrants are diagnosed but do not create driver-race rows.

## Outputs

- `data/processed/driver_race_dataset.csv`
- `data/outputs/notebook_02_validation_report.json`

## Imports

In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Dict, Iterable, Mapping, Optional, Sequence, Set, Tuple

import numpy as np
import pandas as pd

## Configuration

In [2]:
START_YEAR = 2004
END_YEAR = 2025

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "data" / "outputs"

INPUT_PATHS = {
    "schedule": RAW_DIR / "race_schedule.csv",
    "race_results": RAW_DIR / "race_results.csv",
    "qualifying_results": RAW_DIR / "qualifying_results.csv",
}
DATASET_PATH = PROCESSED_DIR / "driver_race_dataset.csv"
VALIDATION_REPORT_PATH = OUTPUT_DIR / "notebook_02_validation_report.json"

DRIVER_RACE_KEY = ["season", "round", "driver_id"]
TEAM_RACE_KEY = ["season", "round", "constructor_id"]
FINISHED_PATTERN = re.compile(r"^\+\d+ Laps?$")

INCIDENT_DNF_STATUSES = {
    "Accident", "Broken wing", "Collision", "Collision damage", "Damage",
    "Debris", "Front wing", "Puncture", "Rear wing", "Spun off",
    "Tyre puncture", "Undertray", "Wheel", "Wheel rim",
}
MECHANICAL_DNF_STATUSES = {
    "Alternator", "Battery", "Brake duct", "Brakes", "Clutch",
    "Cooling system", "Differential", "Driver Seat", "Driveshaft",
    "Drivetrain", "ERS", "Electrical", "Electronics", "Engine",
    "Engine fire", "Engine misfire", "Exhaust", "Fire", "Fuel leak",
    "Fuel pressure", "Fuel pump", "Fuel system", "Gearbox", "Handling",
    "Heat shield fire", "Hydraulics", "Mechanical", "Oil leak",
    "Oil pressure", "Out of fuel", "Overheating", "Pneumatics",
    "Power Unit", "Power loss", "Radiator", "Refuelling", "Seat",
    "Spark plugs", "Steering", "Suspension", "Technical", "Throttle",
    "Track rod", "Transmission", "Turbo", "Tyre", "Vibrations",
    "Water leak", "Water pressure", "Water pump", "Wheel nut",
}
OTHER_DNF_STATUSES = {
    "Illness", "Injured", "Injury", "Not classified", "Retired", "Withdrew",
}
DISQUALIFICATION_STATUSES = {"Disqualified", "Excluded"}

KNOWN_SINGLE_CAR_ENTRIES = {
    (2014, 16, "marussia", "chilton"),
    (2024, 3, "williams", "albon"),
    (2025, 9, "aston_martin", "alonso"),
}

for directory in (PROCESSED_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

## Helper functions

In [3]:
def require_files(paths: Mapping[str, Path]) -> None:
    """Raise a readable error if any required input is absent."""
    missing = [str(path) for path in paths.values() if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing Notebook 1 outputs: {missing}")


def filter_scope(frame: pd.DataFrame) -> pd.DataFrame:
    """Restrict a frame to the configured inclusive season scope."""
    seasons = pd.to_numeric(frame["season"], errors="coerce")
    return frame.loc[seasons.between(START_YEAR, END_YEAR)].reset_index(drop=True)


def classify_finish_status(status: str, laps: int) -> str:
    """Map an API status to one explicit analytical outcome class."""
    if status == "Finished":
        return "finished"
    if status == "Lapped" or FINISHED_PATTERN.fullmatch(status):
        return "lapped_finish"
    if status == "Did not start" or (status == "Withdrew" and laps == 0):
        return "dns"
    if status in DISQUALIFICATION_STATUSES:
        return "disqualified"
    if status in INCIDENT_DNF_STATUSES:
        return "incident_dnf"
    if status in MECHANICAL_DNF_STATUSES:
        return "mechanical_dnf"
    if status in OTHER_DNF_STATUSES:
        return "other_dnf"
    return "unmapped"


def nullable_int_columns(frame: pd.DataFrame, columns: Iterable[str]) -> pd.DataFrame:
    """Apply pandas nullable integer dtype to selected columns."""
    for column in columns:
        if column in frame.columns:
            frame[column] = pd.to_numeric(frame[column], errors="coerce").astype("Int64")
    return frame


def save_csv(frame: pd.DataFrame, path: Path) -> None:
    """Write a CSV atomically so a partial file is never treated as final."""
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary_path, index=False)
    temporary_path.replace(path)


def report_check(name: str, passed: bool, details: Any) -> Dict[str, Any]:
    """Create a JSON-serializable validation record."""
    return {"check": name, "passed": bool(passed), "details": details}


def normalized_csv(frame: pd.DataFrame) -> pd.DataFrame:
    """Normalize values through strings for a CSV round-trip comparison."""
    return frame.reset_index(drop=True).astype("string").fillna("").astype(str)

## Processing

In [4]:
require_files(INPUT_PATHS)

race_schedule = filter_scope(
    pd.read_csv(INPUT_PATHS["schedule"], dtype={"season": "Int64", "round": "Int64"})
)
race_results = filter_scope(pd.read_csv(INPUT_PATHS["race_results"]))
qualifying_results = filter_scope(pd.read_csv(INPUT_PATHS["qualifying_results"]))

race_results = nullable_int_columns(
    race_results,
    ["season", "round", "position", "grid", "laps", "fastest_lap_rank", "fastest_lap_number"],
)
qualifying_results = nullable_int_columns(
    qualifying_results,
    ["season", "round", "qualifying_position"],
)

# Diagnose qualifying entrants who do not have a race-result row.
qualifying_only_rows = (
    qualifying_results.merge(
        race_results[DRIVER_RACE_KEY],
        on=DRIVER_RACE_KEY,
        how="left",
        indicator=True,
    )
    .loc[lambda frame: frame["_merge"].eq("left_only")]
    .drop(columns="_merge")
    .reset_index(drop=True)
)

# Race results are canonical; qualifying is a many-to-one supplemental join.
qualifying_for_join = qualifying_results[
    DRIVER_RACE_KEY + ["constructor_id", "qualifying_position", "q1", "q2", "q3"]
].rename(columns={"constructor_id": "qualifying_constructor_id"})

driver_race_dataset = race_results.merge(
    qualifying_for_join,
    on=DRIVER_RACE_KEY,
    how="left",
    validate="one_to_one",
)
driver_race_dataset["qualifying_missing_flag"] = (
    driver_race_dataset["qualifying_position"].isna().astype("Int64")
)
driver_race_dataset["qualifying_constructor_mismatch_flag"] = (
    driver_race_dataset["qualifying_constructor_id"].notna()
    & driver_race_dataset["qualifying_constructor_id"].ne(driver_race_dataset["constructor_id"])
).astype("Int64")

# Participation and finish-status classification.
driver_race_dataset["finish_status_group"] = [
    classify_finish_status(str(status), int(laps))
    for status, laps in zip(driver_race_dataset["status"], driver_race_dataset["laps"])
]
driver_race_dataset["did_start"] = driver_race_dataset["finish_status_group"].ne("dns").astype("Int64")
driver_race_dataset["did_finish"] = (
    driver_race_dataset["finish_status_group"].isin({"finished", "lapped_finish"}).astype("Int64")
)
driver_race_dataset["is_dnf"] = (
    driver_race_dataset["finish_status_group"].isin(
        {"incident_dnf", "mechanical_dnf", "other_dnf"}
    ).astype("Int64")
)
driver_race_dataset["is_dns"] = driver_race_dataset["finish_status_group"].eq("dns").astype("Int64")
driver_race_dataset["is_disqualified"] = (
    driver_race_dataset["finish_status_group"].eq("disqualified").astype("Int64")
)
driver_race_dataset["dnf_category"] = driver_race_dataset["finish_status_group"].where(
    driver_race_dataset["is_dnf"].eq(1),
    pd.NA,
)

# Field sizes and position features.
driver_race_dataset["field_size"] = (
    driver_race_dataset.groupby(["season", "round"])["driver_id"].transform("size").astype("Int64")
)
driver_race_dataset["qualifying_field_size"] = (
    driver_race_dataset.groupby(["season", "round"])["qualifying_position"]
    .transform("count")
    .astype("Int64")
)
driver_race_dataset["grid_zero_flag"] = driver_race_dataset["grid"].eq(0).astype("Int64")
driver_race_dataset["effective_grid_position"] = driver_race_dataset["grid"].astype("Int64")
pit_lane_starter = driver_race_dataset["grid"].eq(0) & driver_race_dataset["did_start"].eq(1)
driver_race_dataset.loc[pit_lane_starter, "effective_grid_position"] = (
    driver_race_dataset.loc[pit_lane_starter, "field_size"] + 1
)
driver_race_dataset.loc[driver_race_dataset["did_start"].eq(0), "effective_grid_position"] = pd.NA

denominator = (driver_race_dataset["field_size"] - 1).replace(0, pd.NA)
driver_race_dataset["finish_position_pct"] = (
    (driver_race_dataset["position"] - 1) / denominator
)
driver_race_dataset["qualifying_position_pct"] = (
    (driver_race_dataset["qualifying_position"] - 1)
    / (driver_race_dataset["qualifying_field_size"] - 1).replace(0, pd.NA)
)
driver_race_dataset["positions_gained"] = (
    driver_race_dataset["effective_grid_position"] - driver_race_dataset["position"]
)
driver_race_dataset["model_eligible_row"] = (
    driver_race_dataset["did_start"].eq(1)
    & driver_race_dataset["is_disqualified"].eq(0)
    & driver_race_dataset["effective_grid_position"].notna()
    & driver_race_dataset["position"].notna()
).astype("Int64")
driver_race_dataset["driver_full_name"] = (
    driver_race_dataset["given_name"].str.strip()
    + " "
    + driver_race_dataset["family_name"].str.strip()
)

# Teammates are other drivers sharing constructor and race.
team_entry_counts = (
    driver_race_dataset.groupby(TEAM_RACE_KEY)["driver_id"].transform("size").astype("Int64")
)
driver_race_dataset["team_entry_count"] = team_entry_counts
driver_race_dataset["single_car_entry_flag"] = team_entry_counts.eq(1).astype("Int64")

teammate_source = driver_race_dataset[
    DRIVER_RACE_KEY
    + ["constructor_id", "driver_full_name", "position", "effective_grid_position",
       "finish_status_group", "positions_gained"]
].rename(
    columns={
        "driver_id": "teammate_id",
        "driver_full_name": "teammate_name",
        "position": "teammate_finish_position",
        "effective_grid_position": "teammate_effective_grid_position",
        "finish_status_group": "teammate_finish_status_group",
        "positions_gained": "teammate_positions_gained",
    }
)

teammate_pairs = driver_race_dataset[DRIVER_RACE_KEY + ["constructor_id"]].merge(
    teammate_source,
    on=TEAM_RACE_KEY,
    how="left",
)
teammate_pairs = teammate_pairs.loc[
    teammate_pairs["driver_id"].ne(teammate_pairs["teammate_id"])
]
teammate_pairs = teammate_pairs[
    DRIVER_RACE_KEY
    + ["teammate_id", "teammate_name", "teammate_finish_position",
       "teammate_effective_grid_position", "teammate_finish_status_group",
       "teammate_positions_gained"]
]
driver_race_dataset = driver_race_dataset.merge(
    teammate_pairs,
    on=DRIVER_RACE_KEY,
    how="left",
    validate="one_to_one",
)
driver_race_dataset["finish_position_vs_teammate"] = (
    driver_race_dataset["position"] - driver_race_dataset["teammate_finish_position"]
)
driver_race_dataset["positions_gained_vs_teammate"] = (
    driver_race_dataset["positions_gained"] - driver_race_dataset["teammate_positions_gained"]
)

driver_race_dataset = nullable_int_columns(
    driver_race_dataset,
    [
        "season", "round", "position", "grid", "laps", "qualifying_position",
        "field_size", "qualifying_field_size", "effective_grid_position",
        "teammate_finish_position", "teammate_effective_grid_position",
    ],
)
driver_race_dataset = driver_race_dataset.sort_values(
    ["season", "round", "position", "driver_id"]
).reset_index(drop=True)

## Diagnostics

In [5]:
diagnostics = pd.DataFrame(
    {
        "metric": [
            "scheduled races", "driver-race rows", "drivers", "constructors",
            "qualifying matches", "race rows missing qualifying",
            "qualifying-only entrants", "single-car entries",
            "model-eligible rows", "DNFs", "DNS", "disqualifications",
        ],
        "value": [
            len(race_schedule),
            len(driver_race_dataset),
            driver_race_dataset["driver_id"].nunique(),
            driver_race_dataset["constructor_id"].nunique(),
            int(driver_race_dataset["qualifying_missing_flag"].eq(0).sum()),
            int(driver_race_dataset["qualifying_missing_flag"].sum()),
            len(qualifying_only_rows),
            int(driver_race_dataset["single_car_entry_flag"].sum()),
            int(driver_race_dataset["model_eligible_row"].sum()),
            int(driver_race_dataset["is_dnf"].sum()),
            int(driver_race_dataset["is_dns"].sum()),
            int(driver_race_dataset["is_disqualified"].sum()),
        ],
    }
)
display(diagnostics)
display(driver_race_dataset["finish_status_group"].value_counts().rename("rows").to_frame())
display(
    driver_race_dataset.loc[
        driver_race_dataset["single_car_entry_flag"].eq(1),
        ["season", "round", "race_name", "constructor_id", "driver_id", "driver_full_name"],
    ]
)
display(
    qualifying_only_rows[
        ["season", "round", "race_name", "driver_id", "constructor_id", "qualifying_position"]
    ]
)

,metric,value
0,scheduled races,436
1,driver-race rows,9125
2,drivers,111
3,constructors,35
4,qualifying matches,9100
5,race rows missing qualifying,25
6,qualifying-only entrants,5
7,single-car entries,3
8,model-eligible rows,9054
9,DNFs,1731


,rows
finish_status_group,
finished,4621
lapped_finish,2702
mechanical_dnf,799
incident_dnf,706
other_dnf,226
disqualified,37
dns,34


,season,round,race_name,constructor_id,driver_id,driver_full_name
4370,2014,16,Russian Grand Prix,marussia,chilton,Max Chilton
8217,2024,3,Australian Grand Prix,williams,albon,Alexander Albon
8814,2025,9,Spanish Grand Prix,aston_martin,alonso,Fernando Alonso


,season,round,race_name,driver_id,constructor_id,qualifying_position
0,2011,1,Australian Grand Prix,liuzzi,hrt,23
1,2011,1,Australian Grand Prix,karthikeyan,hrt,24
2,2012,1,Australian Grand Prix,rosa,hrt,23
3,2012,1,Australian Grand Prix,karthikeyan,hrt,24
4,2025,9,Spanish Grand Prix,stroll,aston_martin,14


## Save outputs

In [6]:
save_csv(driver_race_dataset, DATASET_PATH)
print(f"Saved {len(driver_race_dataset):,} rows to {DATASET_PATH.relative_to(PROJECT_ROOT)}")

Saved 9,125 rows to data/processed/driver_race_dataset.csv


## Validation

In [7]:
result_keys = set(map(tuple, race_results[DRIVER_RACE_KEY].to_numpy()))
output_keys = set(map(tuple, driver_race_dataset[DRIVER_RACE_KEY].to_numpy()))
actual_single_car_entries = set(
    map(
        tuple,
        driver_race_dataset.loc[
            driver_race_dataset["single_car_entry_flag"].eq(1),
            ["season", "round", "constructor_id", "driver_id"],
        ].to_numpy(),
    )
)
unmapped_statuses = sorted(
    driver_race_dataset.loc[
        driver_race_dataset["finish_status_group"].eq("unmapped"), "status"
    ].unique()
)
team_sizes = driver_race_dataset.groupby(TEAM_RACE_KEY)["driver_id"].size()

teammate_lookup = driver_race_dataset.set_index(DRIVER_RACE_KEY)["teammate_id"]
asymmetric_teammates = []
for row in driver_race_dataset.loc[driver_race_dataset["teammate_id"].notna()].itertuples():
    reverse_key = (row.season, row.round, row.teammate_id)
    if reverse_key not in teammate_lookup.index or teammate_lookup.loc[reverse_key] != row.driver_id:
        asymmetric_teammates.append((row.season, row.round, row.driver_id, row.teammate_id))

expected_columns = {
    "season", "round", "driver_id", "constructor_id", "qualifying_position",
    "grid", "effective_grid_position", "position", "points", "status", "laps",
    "finish_status_group", "did_start", "did_finish", "is_dnf", "is_dns",
    "is_disqualified", "dnf_category", "field_size", "finish_position_pct",
    "positions_gained", "teammate_id", "teammate_finish_position",
    "single_car_entry_flag", "model_eligible_row",
}

checks = [
    report_check(
        "one output row per race-result row",
        len(driver_race_dataset) == len(race_results) and output_keys == result_keys,
        {"input_rows": len(race_results), "output_rows": len(driver_race_dataset)},
    ),
    report_check(
        "unique driver-race keys",
        not driver_race_dataset.duplicated(DRIVER_RACE_KEY).any(),
        int(driver_race_dataset.duplicated(DRIVER_RACE_KEY).sum()),
    ),
    report_check(
        "complete configured season coverage",
        set(driver_race_dataset["season"].astype(int)) == set(range(START_YEAR, END_YEAR + 1)),
        {
            "min": int(driver_race_dataset["season"].min()),
            "max": int(driver_race_dataset["season"].max()),
        },
    ),
    report_check(
        "all finish statuses mapped",
        len(unmapped_statuses) == 0,
        unmapped_statuses,
    ),
    report_check(
        "qualifying join does not change row count",
        len(driver_race_dataset) == len(race_results),
        {
            "matched": int(driver_race_dataset["qualifying_missing_flag"].eq(0).sum()),
            "race_only": int(driver_race_dataset["qualifying_missing_flag"].sum()),
            "qualifying_only": len(qualifying_only_rows),
        },
    ),
    report_check(
        "qualifying constructor IDs agree when present",
        not driver_race_dataset["qualifying_constructor_mismatch_flag"].any(),
        int(driver_race_dataset["qualifying_constructor_mismatch_flag"].sum()),
    ),
    report_check(
        "constructor race groups contain at most two drivers",
        bool(team_sizes.le(2).all()),
        team_sizes.value_counts().sort_index().to_dict(),
    ),
    report_check(
        "known single-car entries match exactly",
        actual_single_car_entries == KNOWN_SINGLE_CAR_ENTRIES,
        {
            "actual": sorted(actual_single_car_entries),
            "expected": sorted(KNOWN_SINGLE_CAR_ENTRIES),
        },
    ),
    report_check(
        "teammate assignments are symmetric",
        len(asymmetric_teammates) == 0,
        asymmetric_teammates,
    ),
    report_check(
        "only single-car entries lack teammate IDs",
        driver_race_dataset["teammate_id"].isna().eq(
            driver_race_dataset["single_car_entry_flag"].eq(1)
        ).all(),
        {
            "missing_teammates": int(driver_race_dataset["teammate_id"].isna().sum()),
            "single_car_entries": int(driver_race_dataset["single_car_entry_flag"].sum()),
        },
    ),
    report_check(
        "effective grid positions are valid for starters",
        driver_race_dataset.loc[
            driver_race_dataset["did_start"].eq(1), "effective_grid_position"
        ].between(1, driver_race_dataset.loc[driver_race_dataset["did_start"].eq(1), "field_size"] + 1).all(),
        {
            "pit_lane_or_unset_starters": int(
                (
                    driver_race_dataset["grid_zero_flag"].eq(1)
                    & driver_race_dataset["did_start"].eq(1)
                ).sum()
            )
        },
    ),
    report_check(
        "model eligibility excludes DNS and disqualifications",
        not (
            driver_race_dataset["model_eligible_row"].eq(1)
            & (
                driver_race_dataset["is_dns"].eq(1)
                | driver_race_dataset["is_disqualified"].eq(1)
            )
        ).any(),
        {"eligible_rows": int(driver_race_dataset["model_eligible_row"].sum())},
    ),
    report_check(
        "expected columns",
        expected_columns <= set(driver_race_dataset.columns),
        sorted(expected_columns - set(driver_race_dataset.columns)),
    ),
]

validation_report = {
    "notebook": "02_build_driver_race_dataset",
    "scope": {"start_year": START_YEAR, "end_year": END_YEAR},
    "status": "PASS" if all(check["passed"] for check in checks) else "FAIL",
    "checks": checks,
}
display(pd.DataFrame(checks))
if validation_report["status"] != "PASS":
    failed = [check for check in checks if not check["passed"]]
    raise AssertionError(f"Validation failed: {failed}")
print("PASS")

,check,passed,details
0,one output row per race-result row,True,"{'input_rows': 9125, 'output_rows': 9125}"
1,unique driver-race keys,True,0
2,complete configured season coverage,True,"{'min': 2004, 'max': 2025}"
3,all finish statuses mapped,True,[]
4,qualifying join does not change row count,True,"{'matched': 9100, 'race_only': 25, 'qualifying..."
5,qualifying constructor IDs agree when present,True,0
6,constructor race groups contain at most two dr...,True,"{1: 3, 2: 4561}"
7,known single-car entries match exactly,True,"{'actual': [(2014, 16, 'marussia', 'chilton'),..."
8,teammate assignments are symmetric,True,[]
9,only single-car entries lack teammate IDs,True,"{'missing_teammates': 3, 'single_car_entries': 3}"


PASS


## Supercheck

In [8]:
saved_dataset = pd.read_csv(DATASET_PATH, dtype=str, keep_default_na=False)
dataset_equal = normalized_csv(driver_race_dataset).equals(normalized_csv(saved_dataset))

superchecks = [
    report_check("driver-race dataset file exists", DATASET_PATH.exists(), str(DATASET_PATH)),
    report_check(
        "saved dataset equals in-memory dataset",
        dataset_equal,
        {"rows": len(saved_dataset), "columns": len(saved_dataset.columns)},
    ),
]
validation_report["superchecks"] = superchecks
validation_report["status"] = (
    "PASS" if all(check["passed"] for check in checks + superchecks) else "FAIL"
)

serializable_report = json.loads(json.dumps(validation_report, default=str))
temporary_report_path = VALIDATION_REPORT_PATH.with_suffix(".json.tmp")
with temporary_report_path.open("w", encoding="utf-8") as handle:
    json.dump(serializable_report, handle, indent=2)
temporary_report_path.replace(VALIDATION_REPORT_PATH)

saved_report = json.loads(VALIDATION_REPORT_PATH.read_text(encoding="utf-8"))
report_equal = saved_report == serializable_report
report_checks = [
    report_check(
        "validation report file exists",
        VALIDATION_REPORT_PATH.exists(),
        str(VALIDATION_REPORT_PATH),
    ),
    report_check("saved validation report equals memory", report_equal, None),
]
display(pd.DataFrame(superchecks + report_checks))
if validation_report["status"] != "PASS" or not all(
    check["passed"] for check in report_checks
):
    raise AssertionError("Supercheck failed")
print("PASS")

,check,passed,details
0,driver-race dataset file exists,True,
1,saved dataset equals in-memory dataset,True,"{'rows': 9125, 'columns': 61}"
2,validation report file exists,True,
3,saved validation report equals memory,True,None


PASS
